# Lesson 6: Essay Writer

Set **`OPENAI_API_KEY`** and **`TAVILY_API_KEY`**. From the project root:

```bash
pip install -r requirements-lesson-06.txt
```

The graph below uses **`with SqliteSaver.from_conn_string(":memory:")`** so the checkpointer lives for that run. The **Gradio** section uses `helper.py`, which opens a SQLite connection that stays valid across UI threads.


## Imports, state, and model


In [ ]:
from dotenv import load_dotenv

load_dotenv()

from typing import List, TypedDict

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, StateGraph
from langchain_openai import ChatOpenAI


class AgentState(TypedDict, total=False):
    task: str
    plan: str
    draft: str
    critique: str
    content: List[str]
    revision_number: int
    max_revisions: int


model = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)


## Prompts, query schema, and Tavily


In [ ]:
import os

from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field
from tavily import TavilyClient

PLAN_PROMPT = """You are an expert writer tasked with writing a high level outline of an essay. \
Write such an outline for the user provided topic. Give an outline of the essay along with any relevant notes \
or instructions for the sections."""

WRITER_PROMPT = """You are an essay assistant tasked with writing excellent 5-paragraph essays.\
Generate the best essay possible for the user's request and the initial outline. \
If the user provides critique, respond with a revised version of your previous attempts. \
Utilize all the information below as needed:

------

{content}"""

REFLECTION_PROMPT = """You are a teacher grading an essay submission. \
Generate critique and recommendations for the user's submission. \
Provide detailed recommendations, including requests for length, depth, style, etc."""

RESEARCH_PLAN_PROMPT = """You are a researcher charged with providing information that can \
be used when writing the following essay. Generate a list of search queries that will gather \
any relevant information. Only generate 3 queries max."""

RESEARCH_CRITIQUE_PROMPT = """You are a researcher charged with providing information that can \
be used when making any requested revisions (as outlined below). \
Generate a list of search queries that will gather any relevant information. Only generate 3 queries max."""


class Queries(BaseModel):
    queries: List[str] = Field(default_factory=list)


tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


## Node functions


In [ ]:
def plan_node(state: AgentState):
    messages = [
        SystemMessage(content=PLAN_PROMPT),
        HumanMessage(content=state["task"]),
    ]
    response = model.invoke(messages)
    return {"plan": response.content}


def research_plan_node(state: AgentState):
    queries = model.with_structured_output(Queries).invoke(
        [
            SystemMessage(content=RESEARCH_PLAN_PROMPT),
            HumanMessage(content=state["task"]),
        ]
    )
    content = state.get("content") or []
    for q in queries.queries:
        response = tavily.search(query=q, max_results=2)
        for r in response["results"]:
            content.append(r["content"])
    return {"content": content}


def generation_node(state: AgentState):
    content = "\n\n".join(state.get("content") or [])
    user_message = HumanMessage(
        content=f"{state['task']}\n\nHere is my plan:\n\n{state['plan']}"
    )
    messages = [
        SystemMessage(content=WRITER_PROMPT.format(content=content)),
        user_message,
    ]
    response = model.invoke(messages)
    return {
        "draft": response.content,
        "revision_number": state.get("revision_number", 1) + 1,
    }


def reflection_node(state: AgentState):
    messages = [
        SystemMessage(content=REFLECTION_PROMPT),
        HumanMessage(content=state["draft"]),
    ]
    response = model.invoke(messages)
    return {"critique": response.content}


def research_critique_node(state: AgentState):
    queries = model.with_structured_output(Queries).invoke(
        [
            SystemMessage(content=RESEARCH_CRITIQUE_PROMPT),
            HumanMessage(content=state["critique"]),
        ]
    )
    content = state.get("content") or []
    for q in queries.queries:
        response = tavily.search(query=q, max_results=2)
        for r in response["results"]:
            content.append(r["content"])
    return {"content": content}


def should_continue(state: AgentState):
    if state.get("revision_number", 0) > state.get("max_revisions", 2):
        return END
    return "reflect"


## Build the graph


In [ ]:
builder = StateGraph(AgentState)
builder.add_node("planner", plan_node)
builder.add_node("generate", generation_node)
builder.add_node("reflect", reflection_node)
builder.add_node("research_plan", research_plan_node)
builder.add_node("research_critique", research_critique_node)
builder.set_entry_point("planner")
builder.add_conditional_edges(
    "generate",
    should_continue,
    {END: END, "reflect": "reflect"},
)
builder.add_edge("planner", "research_plan")
builder.add_edge("research_plan", "generate")
builder.add_edge("reflect", "research_critique")
builder.add_edge("research_critique", "generate")


## Compile, visualize (optional), and stream

Use a **`with SqliteSaver.from_conn_string(...)`** block so the in-memory database stays open for compile + streaming.


In [ ]:
from IPython.display import Image, display

with SqliteSaver.from_conn_string(":memory:") as memory:
    graph = builder.compile(checkpointer=memory)
    try:
        display(Image(graph.get_graph().draw_png()))
    except Exception as e:
        print("Could not render graph PNG:", e)

    thread = {"configurable": {"thread_id": "1"}}
    for s in graph.stream(
        {
            "task": "what is the difference between langchain and langsmith",
            "max_revisions": 2,
            "revision_number": 1,
        },
        thread,
    ):
        print(s)


## Essay writer interface (Gradio)

`helper.py` builds a graph with a long-lived SQLite connection (needed for background threads). Run with **`workbooks/`** as the working directory so `import helper` works.


In [ ]:
import warnings

warnings.filterwarnings("ignore")

from helper import ewriter, writer_gui

MultiAgent = ewriter()
app = writer_gui(MultiAgent.graph)
app.launch()
